# Part 1 - Produce messages to an Apache Kafka topic

In this notebook you will learn how to produce messages to an Apache Kafka topic.

![Produce messages to an Apache Kafka Topic](../img/1%20-%20single%20kafka%20producer%20python.png)

> **📝 NOTE**
>
> _If you haven't yet set up your Kafka credentials, complete [0-setup.ipynb](0-setup.ipynb) first._

## About Kafka topics

Apache Kafka® stores messages in groups called **topics**. The next code cell creates the `pizzas` topic automatically, no console steps needed.

> **📝 NOTE**
> 
> Topics are case-sensitive: `pizzas` is different from `Pizzas`.

To watch messages arrive in real time, open the **Topics → pizzas → Messages** tab in the Aiven Console. If prompted to enable the Kafka REST API, select **Enable**.

## Create the topic and send your first order

The code below:
1. Creates the `pizzas` topic (safe to re-run, skips creation if it already exists)
2. Creates a producer
3. Sends the first pizza order

In [4]:
import json
import os
from dotenv import load_dotenv
from confluent_kafka import Producer
from confluent_kafka.admin import AdminClient, NewTopic

load_dotenv()

KAFKA_SERVICE_URI = os.getenv('KAFKA_SERVICE_URI')

conf = {
    'bootstrap.servers': KAFKA_SERVICE_URI,
    'client.id': 'myclient',
    'security.protocol': 'SSL',
    'ssl.ca.location': '../sslcerts/ca.pem',
    'ssl.certificate.location': '../sslcerts/service.cert',
    'ssl.key.location': '../sslcerts/service.key',
}

# Create the topic (no-op if it already exists)
admin = AdminClient(conf)
for topic, f in admin.create_topics([NewTopic("pizzas", num_partitions=1, replication_factor=1)]).items():
    try:
        f.result()
        print(f"Topic '{topic}' created")
    except Exception as e:
        print(f"Topic '{topic}': {e}")

producer = Producer(conf)

producer.produce(
    "pizzas",
    key=json.dumps({"id": 1}).encode(),
    value=json.dumps({"id": 1, "name": "👨 Francesco", "pizza": "Margherita 🍕"}).encode(),
)
producer.flush()
print("Order sent!")

Topic 'pizzas' created


%4|1785240131.052|TERMINATE|myclient#producer-5| [thrd:app]: Producer terminating with 1 message (79 bytes) still in queue or transit: use flush() to wait for outstanding message delivery


Order sent!


---

## Your Turn! Add more orders

Edit the list below and run the cell. IDs are assigned automatically.

In [5]:
more_orders = [
    {"name": "👩 Adele", "pizza": "Hawaii 🍕+🍍+🥓"},
    {"name": "👦 Jay", "pizza": "Pepperoni 🍕"},
    # Add your own orders here:
    # {"name": "Your Name", "pizza": "Your Favourite 🍕"},
]

for order_id, order in enumerate(more_orders, start=2):
    producer.produce(
        "pizzas",
        key=json.dumps({"id": order_id}).encode(),
        value=json.dumps({"id": order_id, **order}).encode(),
    )

producer.flush()
print(f"{len(more_orders)} orders sent!")

2 orders sent!


%6|1785267075.335|FAIL|myclient#producer-6| [thrd:ssl://142.93.78.5:28284/1]: ssl://142.93.78.5:28284/1: Disconnected: connection reset by peer (after 26944624ms in state UP)
%6|1785267075.381|FAIL|myclient#producer-7| [thrd:ssl://142.93.78.5:28284/1]: ssl://142.93.78.5:28284/1: Disconnected: connection reset by peer (after 26943586ms in state UP)
%6|1785267525.097|FAIL|myclient#producer-7| [thrd:ssl://142.93.78.5:28284/1]: ssl://142.93.78.5:28284/1: Disconnected: connection reset by peer (after 448318ms in state UP, 1 identical error(s) suppressed)
%6|1785267525.221|FAIL|myclient#producer-7| [thrd:ssl://68.183.100.239:28284/2]: ssl://68.183.100.239:28284/2: Disconnected: connection reset by peer (after 449467ms in state UP)
%6|1785267526.630|FAIL|myclient#producer-6| [thrd:ssl://68.183.100.239:28284/2]: ssl://68.183.100.239:28284/2: Disconnected: connection reset by peer (after 450312ms in state UP)
%6|1785267928.444|FAIL|myclient#producer-6| [thrd:ssl://142.93.78.5:28284/1]: ssl://14

> **📝 NOTE**
> We'll need to produce more messages **While the consumer is running** so leave this notebook open

## Great Work! 🥳

We used Python to produce the messages. 

We're currently using the Aiven Console to consume these messages but we can use Python to do the same.

Move onto the [next notebook](2-consume.ipynb) or push the button below

[![Consuming Messages](https://img.shields.io/badge/2-Consuming%20Messages%20With%20Python-a03586?style=for-the-badge&labelColor=ec6147)](./2-consume.ipynb)